In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm
import matplotlib.font_manager as fm
import geopandas as gpd
import scipy.stats as stats
import rasterio
import rioxarray as rxt
import fiona
from glob import glob


%matplotlib qt

In [ ]:
# directory 정의
result_dir = r"D:/ForestFire/CBH/result"
data_dir = r"D:/ForestFire/CBH/data"

### Split raster

In [ ]:
def splitRaster(in_raster, patch_raster_dir, patch_cnt_height, patch_cnt_width, header, reference_crs = rasterio.crs.CRS.from_wkt('PROJCS["Korea 2000 / Unified CS",GEOGCS["KGD2002",DATUM["Korean_Geodetic_Datum_2002",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],\
                                                AUTHORITY["EPSG","6737"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4737"]],PROJECTION["Transverse_Mercator"],\
                                                PARAMETER["latitude_of_origin",38],PARAMETER["central_meridian",127.5],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",1000000],PARAMETER["false_northing",2000000],\
                                                UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5179"]]')
                ):
    print('Processing Starts')
    with rasterio.open(in_raster) as src:
        # reproject the raster
        if src.crs == reference_crs:
            reprojected_raster = in_raster
        else:
            print('Reprojection Starts')
            # calculate the default transform and shape based on the new CRS
            transform, width, height = rasterio.warp.calculate_default_transform(
                src.crs, reference_crs, src.width, src.height, src.bounds
            )
            # update the metadata based on new CRS and Transform
            new_meta = src.meta.copy()
            new_meta.update({
                'crs' : reference_crs,
                'transform': transform,
                'width' : width,
                'height' : height
            })
            # create empty array and reproject
            parent, file = os.path.split(in_raster)
            file_name, format = os.path.splitext(file)
            reprojected_raster = os.path.join(parent, file_name + '_reprojected' + format)
            with rasterio.open(reprojected_raster, 'w', **new_meta) as dst:
                # parameters: original(src) crs & transform, destination(dst) crs & transform, resmapling method
                rasterio.reproject(
                    source = rasterio.band(src, 1),
                    destination = rasterio.band(dst, 1),
                    src_transform = src.transform,
                    src_crs = src.crs,
                    dst_transform = transform,
                    dst_crs = reference_crs,
                    resmapling=Resampling.bilinear
                )
            print('Reprojection Ends')
    
    # Splitting the raster
    with rasterio.open(reprojected_raster) as src:
        print("Splitting Raster into Patches...")
        meta = src.meta.copy()
        width, height = src.width, src.height

        # Ensure minimum patch size
        patch_height = max(1, height // patch_cnt_height) # prevent returning '0', don't use divmod!
        patch_width = max(1, width // patch_cnt_width)

        os.makedirs(patch_raster_dir, exist_ok=True)
        track = 0

        for i in tqdm(range(patch_cnt_height), desc="Processing Rows"):
            for j in tqdm(range(patch_cnt_width), desc="Processing Columns", leave=False):
                window = rasterio.windows.Window(j * patch_width, i * patch_height, patch_width, patch_height) # width and height should be start with '1', not 'zero'
                patch_data = src.read(1, window=window)
                meta.update({
                    "height": patch_height,
                    "width": patch_width,
                    "transform": src.window_transform(window)
                })

                output_patch = os.path.join(patch_raster_dir, header + f"_Patch_{track}.tif")
                with rasterio.open(output_patch, "w", **meta) as dst:
                    dst.write(patch_data, 1)

                track += 1

    print("Processing Ends")
        

In [ ]:
import rasterio
import numpy as np
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
from tqdm import tqdm

def resample_raster(input_raster, reference_raster, output_raster, resampling_method, batch_size=512):
    print('Reading reference raster')
    with rasterio.open(reference_raster) as ref:
        ref_transform = ref.transform
        ref_crs = ref.crs
        ref_width = ref.width
        ref_height = ref.height

    print('Reading target raster')
    with rasterio.open(input_raster) as src:
        new_meta = src.meta.copy()
        new_meta.update({
            "crs": ref_crs,
            "transform": ref_transform,
            "width": ref_width,
            "height": ref_height
        })
        print('Finished reading rasters')

        with rasterio.open(output_raster, "w", **new_meta) as dst:
            print('Starting resampling process...')
            for i in tqdm(range(1, src.count + 1), desc='Processing Bands'):
                for row in tqdm(range(0, ref_height, batch_size), desc=f'Processing Rows for Band {i}', leave=False):
                    for col in range(0, ref_width, batch_size):
                        window = Window(col, row, min(batch_size, ref_width - col), min(batch_size, ref_height - row))
                        data = src.read(i, window=window)
                        destination = np.empty((window.height, window.width), dtype=data.dtype)

                        reproject(
                            source=data,
                            destination=destination,
                            src_transform=src.window_transform(window),
                            src_crs=src.crs,
                            dst_transform=ref_transform,
                            dst_crs=ref_crs,
                            resampling=resampling_method
                        )

                        dst.write(destination, i, window=window)

    print(f"Resampling completed")

### Resampling

In [ ]:
# Resample imsang with DEM
dem_raster = r"F:\CBH\DEM_merge_5179.tif"
imsang_raster = r"F:\CBH\Imsang_Raster2.tif"
imsang_reproject = r"F:\CBH\Imsang_Raster2_Resample_Nearest3.tif"
resample_raster(imsang_raster, dem_raster, imsang_reproject, Resampling.nearest, batch_size=512)

In [ ]:
dem_raster = r"G:/CBH/DEM_merge_5179.tif"
imsang_reproject = r"F:\CBH\Imsang_Raster2_Float_Clip.tif"
with rasterio.open(dem_raster) as src:
    print(src.height, src.width)
    # Read imsang raster and split
    header = 'Imsang'
    out_raster_dir = r"D:/ForestFire/CBH/data"
    patch_cnt_height = 5
    patch_cnt_width = 5
    splitRaster(imsang_reproject, out_raster_dir, patch_cnt_height, patch_cnt_width, header, src.crs)

### Split Raster

In [ ]:
# split raster - Execution code
dem_raster = r"F:/CBH/DEM_merge_5179.tif"

header_lst = ['DEM'] # 'Slope', 'Aspect', 
raster_lst = [r"F:/CBH/DEM_merge_5179.tif"] # r"D:/ForestFire/CBH/data/Slope_rad_5179.tif", 
out_raster_dir = r"D:/ForestFire/CBH/data"
patch_cnt_height = 5
patch_cnt_width = 5
with rasterio.open(dem_raster) as src:
    for header, in_raster in zip(header_lst, raster_lst):
        splitRaster(in_raster, out_raster_dir, patch_cnt_height, patch_cnt_width, header, src.crs)

# Make CR, CBH raster

In [ ]:
# 함수정의
def func3(H, D, EL, SL, CD, a, b1, b2, b3, c1, d1, d2, d3, d4):
    """Vectorized version of func3"""
    H_log, D_log = np.log1p(H), np.log1p(D)
    size = (b1 * H_log / D_log) + (b2 * H_log) + (b3 * D_log**2)
    comp = c1 * CD
    site = (d1 * EL) + (d2 * EL**2) + (d3 * SL) + (d4 * SL**2)
    x = size + comp + site + a
    cr = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    return cr

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
from tqdm import tqdm
# Same as the height & width of the spliting processing
patch_cnt_height = 5
patch_cnt_width = 5
# Load parameters
params = pd.read_csv(os.path.join(result_dir, "CR_Han_Params_FIN.csv"), encoding='cp949')  # SID, Species, Parameters
params_arr = np.array(params.iloc[:, [i for i in range(len(params.columns)) if not i == 1]])  # SID, Parameters
params_dict = {int(params_arr[idx, 0]): params_arr[idx, 1:] for idx in range(len(params_arr))}
print('Creating parameters dictionary ends')

# Load tree input parameters
tree_input = np.loadtxt(os.path.join(result_dir, 'imsang_params5.txt'), dtype='float')  # imsang_index, koftr_cd, dbh, height, crown density, elevation, slope
print('Getting tree-level inputs Ends')

# Iterate patches: Test PATCH-1
start = 9
for patch_num in tqdm(range(start, patch_cnt_height * patch_cnt_width), desc='Patch Processing'): # patch_cnt_height * patch_cnt_width
    dem_raster = os.path.join(data_dir, f"DEM_Patch_{patch_num}.tif")
    slope_raster = os.path.join(data_dir, f"Slope_Patch_{patch_num}.tif")
    imsang_raster = os.path.join(data_dir, f"Imsang_Patch_{patch_num}.tif")

    # Load raster data
    with rasterio.open(imsang_raster) as imsang_ras, \
         rasterio.open(dem_raster) as dem_ras, \
         rasterio.open(slope_raster) as slope_ras:
             
        print("Imsang Raster Shape:", imsang_ras.shape)  
        print("DEM Raster Shape:", dem_ras.shape)
        print("Slope Raster Shape:", slope_ras.shape)

        # Read and flatten bands
        imsang_bnd = imsang_ras.read(1).flatten()
        dem_bnd = dem_ras.read(1).flatten()
        slope_bnd = np.tan(np.radians(slope_ras.read(1))).flatten()

        # Get metadata
        ref_meta = imsang_ras.meta.copy()
        ref_height, ref_width = imsang_ras.height, imsang_ras.width
        imsang_nodata, dem_nodata, slope_nodata = imsang_ras.nodata, dem_ras.nodata, slope_ras.nodata
             
    print('Reading Rasters Ends')

    # Apply vectorized masking
    valid_mask = (imsang_bnd != imsang_nodata) & (dem_bnd != dem_nodata) & (slope_bnd != slope_nodata)
    if not any(valid_mask):
        pass
        print(f'No valid data in Patch{patch_num}')
    else:
        # Get matching indices for tree_input
        valid_imsang_bnd = imsang_bnd[valid_mask]
        print('Start to get matching indices')
        indices = np.searchsorted(tree_input[:, 0], valid_imsang_bnd) # 데이터가 존재하는 임상도의 밴드 내 sid와 tree_input 내 sid가 일치하는 index값을 찾아서 오름차순으로 나열..?
        print('Matching indices Ends')
    
        # Extract corresponding values
        tree_df = pd.DataFrame(tree_input[:, 1:6], index=tree_input[:, 0].astype(int))
        tree_df.columns = ['species', 'dbh', 'height', 'cd', 'elevation']  # if needed
        print('Create DF')
        # Reindex with valid_imsang_bnd (which are SIDs)
        matched = tree_df.reindex(valid_imsang_bnd.astype(int))  # This will insert NaN where no match
        matched = matched.fillna(-99)
        print('Reindexing with nan value')
        # Extract values (automatically aligned)
        sid = matched['species'].to_numpy()
        dbh = matched['dbh'].to_numpy()
        height = matched['height'].to_numpy()
        cd = matched['cd'].to_numpy()
        print('Extract the necessary values')
        elevation = dem_bnd[valid_mask]
        slope = slope_bnd[valid_mask]
    
        # Get parameters from `params_dict`
        dict_len = len(next(iter(params_dict.values())))
        params_array = np.array([
            params_dict.get(int(s), [np.nan] * dict_len) if s != -99. else [np.nan] * dict_len
            for s in sid
        ])
        # dictionary 내 모든 행의 길이가 dict_len과 일치하는지 디버깅(Debugging)
        assert all(len(params_dict[k]) == dict_len for k in params_dict), "Inconsistent lengths of parameters' dictionray!"
    
        # Vectorized function callr
        cr = np.full(imsang_bnd.shape, np.nan, dtype=np.float32)
        cbh = np.full(imsang_bnd.shape, np.nan, dtype=np.float32)
        
        print('Calculation of CR & CBH Starts')
        cr[valid_mask] = func3(height, dbh, elevation, slope, cd, *params_array.T)
        cbh[valid_mask] = (1 - cr[valid_mask]) * height
        print('Calculation Ends')
    
        # Reshape to original raster shape
        cr_arr = cr.reshape(ref_height, ref_width)
        cbh_arr = cbh.reshape(ref_height, ref_width)
    
        # Save the new raster
        ref_meta.update({
            'count': 1,
            'dtype': np.float32,
            'nodata': np.nan
        })
    
        out_cr = os.path.join(result_dir, f'CR_Patch_{patch_num}.tif')
        out_cbh = os.path.join(result_dir, f'CBH_Patch_{patch_num}.tif')
    
        print('Start saving the CR into the raster')
        with rasterio.open(out_cr, 'w', **ref_meta) as dst1:
            dst1.write(cr_arr, 1)
        print('Start saving the CBH into the raster')
        with rasterio.open(out_cbh, 'w', **ref_meta) as dst2:
            dst2.write(cbh_arr, 1)
    

print("Processing completed")


# Mosaic Rasters

In [ ]:
from rasterio.merge import merge

In [ ]:
# mosaic raster
cbh_raster_lst = glob(os.path.join(result_dir, 'CBH_Patch_*.tif'))
cr_raster_lst = glob(os.path.join(result_dir, 'CR_Patch_*.tif'))
`
cbh_raster_files = [rasterio.open(ras) for ras in cbh_raster_lst]
print('Reading CBH patches Ends')
cr_raster_files = [rasterio.open(ras) for ras in cr_raster_lst]
print('Reading CR patches Ends')
mosaic_cbh, cbh_transform = merge(cbh_raster_files)
print('Merging CBH patches Ends')
mosaic_cr, cr_transform = merge(cr_raster_files)
print('Merging CR patches Ends')

cbh_meta = cbh_raster_files[0].meta.copy()
cr_meta = cr_raster_files[0].meta.copy()

In [ ]:
cbh_meta.update({
    'driver' : 'GTiff',
    'height' : mosaic_cbh.shape[1],
    'width' : mosaic_cbh.shape[2],
    'transform' : cbh_transform,
    'nodata' : cbh_raster_files[0].nodata,
})

cr_meta.update({
    'driver' : 'GTiff',
    'height' : mosaic_cr.shape[1],
    'width' : mosaic_cr.shape[2],
    'transform' : cr_transform,
    'nodata' : cr_raster_files[0].nodata,
})
print('Updating CBH & CR metadata Ends')

out_cbh = os.path.join(result_dir, 'CBH_Mosaic.tif')
out_cr = os.path.join(result_dir, 'CR_Mosaic.tif')

with rasterio.open(out_cbh, 'w', **cbh_meta) as dst:
    dst.write(mosaic_cbh)

print('Saving CBH raster Ends')

with rasterio.open(out_cr, 'w', **cr_meta) as dst:
    dst.write(mosaic_cr)

print('Saving CR raster Ends')


# close the files
for file in cbh_raster_lst:
    file.close()

for file in cr_raster_lst:
    file.close()

print('Close the files Ends')
print('Whole process Ends')

# Draw Distribution

In [ ]:
# set korean font
import matplotlib.font_manager as fm
import matplotlib as mpl

# 예: 나눔고딕 또는 맑은고딕 설정
font_path = "C:/Windows/Fonts/malgun.ttf"  # Windows: 맑은고딕
# font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"  # Linux

font_name = fm.FontProperties(fname=font_path).get_name()
mpl.rc('font', family=font_name)

# 한글 깨짐 방지 (마이너스 부호 처리)
mpl.rcParams['axes.unicode_minus'] = False

In [ ]:
# 분포
import rasterio
import numpy as np
import random

sample_size = 5000
samples = []

with rasterio.open(os.path.join(result_dir, 'CBH_Mosaic_arc1.tif')) as src:
    band = src.read(1, masked=True)  # Use masked=True to ignore no-data

    # Get only valid data to sample from
    valid_data = band.compressed()  # removes masked (no-data) values

    # Random sample without loading entire raster if possible
    if len(valid_data) > sample_size:
        samples = np.random.choice(valid_data, sample_size, replace=False)
    else:
        samples = valid_data

# Plot
sns.displot(samples, bins=50, kde=True)
plt.xlabel('CBH (m)')
plt.show()


In [ ]:
len(samples[samples == 0])

In [ ]:
# NFI 지하고
parent_dir = r"D:/ForestFire/CBH"
df = pd.read_csv(os.path.join(parent_dir, r'data/NFI7-Immok-Filtered3.csv'))
sns.displot(df['지하고'], bins=50, kde=True)
plt.xlabel('CBH(cm)')

### notes

In [ ]:
tree_df = pd.DataFrame(tree_input[:, 1:6], index=tree_input[:, 0].astype(int))
tree_df.columns = ['species', 'dbh', 'height', 'cd', 'elevation']  # if needed
print('Create DF')
# Reindex with valid_imsang_bnd (which are SIDs)
matched = tree_df.reindex(valid_imsang_bnd.astype(int))  # This will insert NaN where no match
matched = matched.fillna(-99)
print('Reindexing with nan value')
# Extract values (automatically aligned)
sid = matched['species'].to_numpy()
dbh = matched['dbh'].to_numpy()
height = matched['height'].to_numpy()
cd = matched['cd'].to_numpy()
print('Extract the necessary values')

In [ ]:
dict_len = len(next(iter(params_dict.values())))
params_array = np.array([
    params_dict.get(int(s), np.nan) if s != -99. else [np.nan] * dict_len
    for s in sid
])
len(params_array)

In [ ]:
len(valid_imsang_bnd)

In [ ]:
len(next(iter(params_dict.values())))

In [ ]:
tree_dict = {key: row[1:5] for key, row in zip(tree_input[:, 0], tree_input)}

# Initialize result array with NaNs
result = np.full((len(indices), 4), np.nan)

# Fill result with matched values
for i, key in tqdm(enumerate(indices), total=len(indices)):
    if key in tree_dict:
        result[i] = tree_dict[key]


In [ ]:
tree_input

In [ ]:
sid = tree_input[indices, 1]
height = tree_input[indices, 3]
dbh = tree_input[indices, 2]
cd = tree_input[indices, 4]
elevation = dem_bnd[valid_mask]
slope = slope_bnd[valid_mask]

In [ ]:
np.unique(indices)

In [ ]:
tree_input[tree_input[:, 0] == 2552161]

In [ ]:
# save the result into the raster
for ras_idx in tqdm(range(ras_start_idx, ras_end_idx), desc="Calculating Crown Height", leave=True):
    reference_name = f"Compsoited_elev-dem-id{i}.tif"
    output_raster = os.path.join(result_dir, f"CBH_{i}.tif")
    # get the metadata from the reference raster
    with rasterio.open(os.path.join(data_dir, reference_name)) as ref_src:
        ref_crs = ref_src.crs 
        ref_transform = ref_src.transform
        ref_bounds = ref_src.bounds                    
        ref_meta = ref_src.meta.copy()
        height, width = ref_src.height, ref_src.width  # Get raster dimensions
        print('Reading the reference data Ends')

    # Update the meta data
    ref_meta.update({
        'count' : 2,
        'dtype' : np.float32,
        'nodata' : np.nan
    })

    # Save the new raster
    with rasterio.open(output_raster, 'w', **ref_meta) as dst:
        dst.write(ch_value_dict[ras_dix], 1)
        dst.write(cbh_value_dict[ras_dix], 2)

# Others

In [ ]:
from copy import deepcopy

In [ ]:
# 임상도 불러오기
# 임상도 불러오기
gdb_dir = r"G:\CBH\Imsang_merge.gdb"
if fiona.listlayers(gdb_dir):
    layer = fiona.listlayers(gdb_dir)
    imsang_shape = gpd.read_file(os.path.join(gdb_dir), layer=layer[0])

In [ ]:
# 임상밴드에 수종코드 composite
idx = [i for i in imsang_shape.index]
sid = imsang_shape[['KOFTR_GROU']].values.ravel()
sid_dict = {i : int(j) for i, j in zip(idx, sid) if idx != -2147483647}
sid_bnd = deepcopy(imsang_bnd)
for key, val in sid_dict.items():
    sid_bnd[imsang_bnd == key] = value
if len(imsang_bnd) == len(dem_bnd) == len(slope_bnd):
    print('The length matches. Start to stack the bands')
    feature_arr = np.vstack((imsang_bnd, dem_bnd, slope_bnd))

In [ ]:
# Vectorize -> memory inefficiency problem
def replace_value(x):
    if x == -2147483647:
        return np.nan
    return sid_dict[x] if x in sid_dict else np.nan

# Vectorize the function for 2d array
vectorized_replace_value = np.vectorize(replace_value)
# Apply the vectorized function to the 2D array
sid_bnd = vectorized_replace_value(imsang_bnd)

In [ ]:
container = []
line_length = np.shape(data[0, 2:])
for i in tqdm(idx):
    if i in data[:, 1]: container.append(data[i, 2:])
    else: container.append(np.nan((1, line_length)))
concat_data = np.array(container)

In [ ]:
with rasterio.open(dem_raster) as src:
    dem_meta = src.meta.copy()
    dem_crs = src.crs
    dem_transform = src.transform
    dem_shape = (src.height, src.width)
print('Read information from DEM raster')

# 임상도 rasterize
shapes1 = [(geom, v1) for geom, v1 in zip(imsang.geometry, imsang.ID)]
shapes2 = [(geom, v2) for geom, v2 in zip(imsang.geometry, imsang.KOFTR_GROU)]
rasterized_imsang1 = rasterio.features.rasterize(
    shapes = shapes1,
    out_shape = dem_shape,
    transform = dem_transform,
    fill=np.nan,
    dtype = rasterio.int32
)
"""rasterized_imsang2 = rasterio.features.rasterize(
    shapes = shapes2,
    out_shape = dem_shape,
    transform = dem_transform,
    fill=-99,
    dtype = rasterio.int32
)
print('Rasterize imsang shapefile')"""

# save the rasterized imsang shape
imsang_raster = os.path.join(data_dir, 'imsang_rasterized.tif')
dem_meta.update({'count':1, 'dtype': rasterio.int32, 'nodata':-99.})
with rasterio.open(imsang_raster, 'w', **dem_meta) as dst:
    dst.write(rasterized_imsang1, 1)
    # dst.write(rasterized_imsang, 2)

print('Save imsang raster file')

# visuzlie new imsang raster
with rasterio.open(imsang_raster) as src:
    ras_data = src.read(2)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
plt.figure(figsize=(8, 6))
plt.imshow(ras_data, cmap='blue')
plt.colorbar(label='Species')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

### Preprocessing

In [ ]:
# imsang 기준 특성값 데이터 불러오기
# features: imsang_index, koftr_cd, dbh, height, crown density, elevation, slope
data = np.loadtxt(os.path.join(result_dir, 'imsang_params.txt'), dtype='float')

In [ ]:
ras_start_idx = 10
ras_end_idx = 15
result_dict = {i : 0 for i in range(ras_start_idx, ras_end_idx)}
for i in tqdm(range(ras_start_idx, ras_end_idx), desc='Raster', leave=True):
    file_name = f"Compsoited_elev-dem-id{i}.tif"
    with rasterio.open(os.path.join(data_dir, file_name)) as src:
        band1 = src.read(1)
        band2 = src.read(2)
        band3 = src.read(3)
    print('Reading the band ends')

    # replace nodata value to np.nan
    band1 = np.where((band1 == -2147483648.0) | (band1 ==-2147483647.0), np.nan, band1)
    band2 = np.where((band2 == 3.4e+38), np.nan, band2)
    band3 = np.where((band3 == 3.3999999521443642e+38), np.nan, band3)
    print('Replacing nodata value to NAN Ends')
    
    # Convert band1 to a 1D array for efficient processing
    band1_flat = band1.ravel()
    print('Flatten the band')
    
    # Extract valid values from band1
    valid_mask = ~np.isnan(band1_flat) 
    valid_values = band1_flat[valid_mask]
    print('Extract only valid values')
    
    
    # make a array to save values
    s_values = np.full_like(band1_flat, np.nan, dtype=np.int32)
    dbh_values = np.full_like(band1_flat, np.nan, dtype=np.float32)
    h_values = np.full_like(band1_flat, np.nan, dtype=np.float32)
    cr_values = np.full_like(band1_flat, np.nan, dtype=np.float32)
    print('Extract corresponding Species, DBH, H, and CR values')

    # find the mathcing indices using the imsang_index
    matching_indices = np.searchsorted(data[:, 0], valid_values)
    valid_matches = (data[matching_indices, 0] == valid_values)
    valid_indices = valid_mask.nonzero()[0] 
    print('Finding the matching indices Ends')

    # save the value based on the mathcing indices
    s_values[valid_indices[valid_matches]] = data[matching_indices[valid_matches], 1]
    print('Extract the value from matching indices Ends: Species')
    dbh_values[valid_indices[valid_matches]] = data[matching_indices[valid_matches], 2]
    print('Extract the value from matching indices Ends: DBH')
    h_values[valid_indices[valid_matches]] = data[matching_indices[valid_matches], 3]
    print('Extract the value from matching indices Ends: Height')
    cr_values[valid_indices[valid_matches]] = data[matching_indices[valid_matches], 4]
    print('Extract the value from matching indices Ends: Crown Density')
    
    # # instead of np.interp(interpoaltion) -> use for function and dictionary to map the values
    # data_dict = {int(line[0]): (line[1], line[2], line[3], line[4]) for line in data}
    # for i, value in tqdm(enumerate(valid_values), total=len(valid_values), desc='Mapping the values', leave=False):
    #     if int(value) in data_dict:
    #         s_values[valid_mask][i], dbh_values[valid_mask][i], h_values[valid_mask][i], cr_values[valid_mask][i] = data_dict[int(value)]

    
    # Reshape back to original raster shape
    s_out = s_values.reshape(band1.shape)
    dbh_out = dbh_values.reshape(band1.shape)
    h_out = h_values.reshape(band1.shape)
    cr_out = cr_values.reshape(band1.shape)
    print('Reshaping Ends')
    # column header: index of imsang, species code, elevation, slope, dbh, height, crown density
    result_dict[i] = np.stack((band1, s_out, band2, band3, dbh_out, h_out, cr_out), axis=0)

In [ ]:
# check the value has been saved properly
# Extract raster data
temp = result_dict[11][4, :, :]

# Identify NoData value
no_data_val = temp[0][0]
print(no_data_val)

# Find indices where NoData (NaN) does NOT exist
valid_indices = np.where(~np.isnan(temp)) # type(valid_indices) == 'tuple' # np.where(temp != no_data_val)  # 

# Extract corresponding values
valid_values = temp[valid_indices]

print("Valid Indices:", valid_indices)  # Rows and columns of valid data
print("Valid Values:", valid_values)  # Actual values

In [ ]:
temp

# Call the parameters and Calculate CR & CBH

In [ ]:
# parameters
params_set = pd.read_csv(os.path.join(result_dir, 'CR_Han_result_LogTrans_NoAZ_CV_FIN.csv'), encoding='cp949')
params_dict = {params_set.loc[i, 'SID'] : list(params_set.iloc[i, [i for i in range(-9, 0)]]) for i in params_set.index }

In [ ]:
# funciton with log transformation of DBH & Height
def func2(X, *params):
    a, b1, b2, b3, c1, d1, d2, d3, d4, d5, d6 = params
    H, D, EL, SL, AZ, CD = X
    H_log, D_log = np.log1p(H), np.log1p(D)
    size = (b1 * H_log/D_log)+(b2 * H_log)+(b3 * D_log**2)
    comp = c1 * CD
    site = (d1 * EL)+(d2 * EL**2) + (d3 * SL)+(d4 * SL**2)+(d5 * SL * np.sin(AZ))+(d6 * SL * np.cos(AZ))
    x = size + comp + site + a
    cr = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    return cr

In [ ]:
result_dict[11][:, 1, 1]

In [ ]:
# Start and end indices for raster processing (10-15)
ras_start_idx = 11
ras_end_idx = 12

# Dictionary to save CR values for each raster index
ch_value_dict = {}
cbh_value_dict = {}

# Process each raster
for ras_idx in tqdm(range(ras_start_idx, ras_end_idx), desc="Calculating Crown Height", leave=True):
    raster_data_arr = result_dict[ras_idx]  # Shape: (bands, height, width)

    # Extract raster dimensions
    height, width = raster_data_arr.shape[1], raster_data_arr.shape[2]

    # Create output array for CR values
    cr_values = np.full((height, width), np.nan, dtype=np.float32)
    cbh_values = np.full((height, width), np.nan, dtype=np.float32)
    print('Creating array to save the results Ends')
    
    # Extract input values for processing (Bands 2 and beyond)
    input_values = raster_data_arr[2:].copy()  # Shape: (num_bands-2, height, width)

    # Handle NaN values: Create mask for valid pixels
    mask = ~np.isnan(input_values[0])  # True where valid, False where NaN

    # Convert elevation (band index 2) from cm to meters
    input_values[0] *= 0.01  # Element-wise operation (vectorized)

    # Clip slope values (band index 3) to max 400%
    input_values[1] = np.clip(input_values[1], None, 400)  # Faster than if-else

    # Extract species band (index 1)
    species = raster_data_arr[1]

    # Process only valid pixels (apply function in bulk)
    valid_species = species[mask]
    valid_inputs = input_values[:, mask]  # Extract valid pixel data only
    print('Extracting the inputs without no data Ends')
    
    # Retrieve model parameters for each species
    params = np.array(np.array([params_dict.get(sp, np.nan)) for sp in valid_species])
    print('Getting the parameters for each species Ends')
    
    # Compute predicted CR values for valid pixels
    print('Computing CR Starts')
    pred_ch = np.array([func2(valid_inputs[:, i], params[i]) for i in range(len(valid_species))])
    print('Computing CR Ends')
    
    # Assign computed values back into `cr_values`
    cr_values[mask] = pred_ch
    cbh_values[mask] = pred_ch * raster_data_arr[5][mask]
    print('Computing CBH Ends')
    
    # Save results in dictionary
    ch_value_dict[ras_idx] = cr_values
    cbh_value_dict[ras_dix] = cbh_values
    print('Saving the results Ends')

In [ ]:
# temporary code for the test
# Retrieve model parameters for each species
params =[np.array(params_dict.get(sp, np.nan)).flatten() for sp in valid_species]
print('Getting the parameters for each species Ends')

# Compute predicted CR values for valid pixels
print('Computing CR Starts')
pred_cr = np.array([func2(valid_inputs[:, i], params[i]) for i in range(len(valid_species))])
print('Computing CR Ends')

# Assign computed values back into `cr_values`
cr_values[mask] = pred_cr
cbh_values[mask] = pred_cr * raster_data_arr[5][mask]
print('Computing CBH Ends')

# Save results in dictionary
ch_value_dict[ras_idx] = cr_values
cbh_value_dict[ras_dix] = cbh_values
print('Saving the results Ends')

In [ ]:
input_values[0]

In [ ]:
params_dict.get(11, np.nan)

In [ ]:
np.array(params_dict.get(16, np.nan)).flatten()

In [ ]:
# save the result into the raster
for ras_idx in tqdm(range(ras_start_idx, ras_end_idx), desc="Calculating Crown Height", leave=True):
    reference_name = f"Compsoited_elev-dem-id{i}.tif"
    output_raster = os.path.join(result_dir, f"CBH_{i}.tif")
    # get the metadata from the reference raster
    with rasterio.open(os.path.join(data_dir, reference_name)) as ref_src:
        ref_crs = ref_src.crs 
        ref_transform = ref_src.transform
        ref_bounds = ref_src.bounds                    
        ref_meta = ref_src.meta.copy()
        height, width = ref_src.height, ref_src.width  # Get raster dimensions
        print('Reading the reference data Ends')

    # Update the meta data
    ref_meta.update({
        'count' : 2,
        'dtype' : np.float32,
        'nodata' : np.nan
    })

    # Save the new raster
    with rasterio.open(output_raster, 'w', **ref_meta) as dst:
        dst.write(ch_value_dict[ras_dix], 1)
        dst.write(cbh_value_dict[ras_dix], 2)

## Visualize

In [ ]:
# check the distirbution of crown ratio
plt.hist(ch_value_dict[ras_dict][~np.isnan(ch_value_dict)])
plt.title('Distribution of the crown ratio')

In [ ]:
# check the distirbution of crown base height
plt.hist(cbh_value_dict[ras_dict][~np.isnan(cbh_value_dict)])
plt.title('Distribution of the crown base height')